In [68]:
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "raw_data"
REPO_ROOT = PROJECT_DIR.parent
ENV_FILE = REPO_ROOT / ".env"

SPOTIFY_KAGGLE_CSV = DATA_DIR / "spotify_kaggle.csv"
SPOTIFY_KAGGLE_REFINE_CSV = DATA_DIR / "spotify_kaggle_refine.csv"

df = pd.read_csv(SPOTIFY_KAGGLE_CSV, index_col=False)
df.columns

Index(['energy', 'tempo', 'danceability', 'playlist_genre', 'loudness',
       'liveness', 'valence', 'track_artist', 'time_signature', 'speechiness',
       'track_popularity', 'track_href', 'uri', 'track_album_name',
       'playlist_name', 'analysis_url', 'track_id', 'track_name',
       'track_album_release_date', 'instrumentalness', 'track_album_id',
       'mode', 'key', 'duration_ms', 'acousticness', 'id', 'playlist_subgenre',
       'type', 'playlist_id'],
      dtype='str')

In [69]:
useless_column = [
    'playlist_name','track_album_name','track_album_release_date','track_href', 
    'id', 'analysis_url', 'track_album_id', 'playlist_id']

df.drop(columns = useless_column, inplace=True)
df.head(1)

,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,uri,track_id,track_name,instrumentalness,mode,key,duration_ms,acousticness,playlist_subgenre,type
0,0.592,157.969,0.521,pop,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",3,0.0304,...,spotify:track:2plbrEY59IikOBgBGLjaoe,2plbrEY59IikOBgBGLjaoe,Die With A Smile,0.0,0,6,251668,0.308,mainstream,audio_features


In [70]:
first_genre = df["playlist_genre"].fillna("").astype(str).str.strip()
second_genre = df["playlist_subgenre"].fillna("").astype(str).str.strip()
df["genre"] = first_genre.where(second_genre.eq(""), first_genre + "," + second_genre)
df.drop(columns=["playlist_genre","playlist_subgenre"], inplace=True)
df.head(1)

,energy,tempo,danceability,loudness,liveness,valence,track_artist,time_signature,speechiness,track_popularity,uri,track_id,track_name,instrumentalness,mode,key,duration_ms,acousticness,type,genre
0,0.592,157.969,0.521,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",3,0.0304,100,spotify:track:2plbrEY59IikOBgBGLjaoe,2plbrEY59IikOBgBGLjaoe,Die With A Smile,0.0,0,6,251668,0.308,audio_features,"pop,mainstream"


In [71]:
dup = df[df["track_id"].duplicated(keep=False)].sort_values("track_id")
cols = ["track_id", "track_name", "track_artist", "genre", "track_popularity"]
dup[cols]

,track_id,track_name,track_artist,genre,track_popularity
192,01Lr5YepbgjXAWR9iOEyH1,Love Sosa,Chief Keef,"hip-hop,gangster",77
226,01Lr5YepbgjXAWR9iOEyH1,Love Sosa,Chief Keef,"hip-hop,trap",77
135,05RgAMGypEvqhNs5hPCbMS,Panama - 2015 Remaster,Van Halen,"rock,classic",72
641,05RgAMGypEvqhNs5hPCbMS,Panama - 2015 Remaster,Van Halen,"rock,80s",72
411,07nH4ifBxUB4lZcsf44Brn,Blame (feat. John Newman),"Calvin Harris, John Newman","electronic,modern",79
...,...,...,...,...,...
392,7wBThXx7BGZHJJ3aN3OPvv,Confessions Part II,USHER,"r&b,modern",68
1217,7wZUrN8oemZfsEd1CGkbXE,Bleeding Love,Leona Lewis,"blues,classic",75
868,7wZUrN8oemZfsEd1CGkbXE,Bleeding Love,Leona Lewis,"pop,soft",75
559,7ySC0IjVS1PMEdsZOvsUK2,OZEBA,Rema,"afrobeats,african",68


서로 다른 플레이리스트에 같은 노래가 속해있는 경우가 있어서 인기도 기준으로 가장 높은 것만 남기고 제거

In [72]:
df = (
    df.sort_values("track_popularity", ascending=False)
    .drop_duplicates(subset="track_id", keep="first")
    .reset_index(drop=True)
)

df["track_id"].duplicated().sum()

np.int64(0)

In [73]:
df["type"] = "music"
df.head(10)

,energy,tempo,danceability,loudness,liveness,valence,track_artist,time_signature,speechiness,track_popularity,uri,track_id,track_name,instrumentalness,mode,key,duration_ms,acousticness,type,genre
0,0.592,157.969,0.521,-7.777,0.1220,0.535,"Lady Gaga, Bruno Mars",3,0.0304,100,spotify:track:2plbrEY59IikOBgBGLjaoe,2plbrEY59IikOBgBGLjaoe,Die With A Smile,0.000000,0,6,251668,0.3080,music,"pop,global"
1,0.783,149.027,0.777,-4.477,0.3550,0.939,"ROSÉ, Bruno Mars",4,0.2600,98,spotify:track:5vNRhkKd0yEAg8suGBpjeY,5vNRhkKd0yEAg8suGBpjeY,APT.,0.000000,0,0,169917,0.0283,music,"pop,global"
2,0.507,104.978,0.747,-10.171,0.1170,0.438,Billie Eilish,4,0.0358,97,spotify:track:6dOtVTDdiauQNBQEDOtlAB,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,0.060800,1,2,210373,0.2000,music,"pop,mainstream"
3,0.582,116.712,0.700,-5.960,0.0881,0.785,Chappell Roan,4,0.0356,94,spotify:track:0WbMK4wrZ1wFSty9F7FCgu,0WbMK4wrZ1wFSty9F7FCgu,"Good Luck, Babe!",0.000000,0,11,218424,0.0502,music,"pop,mainstream"
4,0.668,128.027,0.924,-6.795,0.0678,0.787,KAROL G,4,0.0469,93,spotify:track:6WatFBLVB0x077xWeoVc2k,6WatFBLVB0x077xWeoVc2k,Si Antes Te Hubiera Conocido,0.000594,1,11,195824,0.4460,music,"pop,mainstream"
5,0.651,112.648,0.694,-6.968,0.0787,0.471,"Oscar Maydon, Fuerza Regida",3,0.0784,93,spotify:track:1cOboCuWYI2osTOfolMRS6,1cOboCuWYI2osTOfolMRS6,Tu Boda,0.000014,1,6,225881,0.4960,music,"pop,mainstream"
6,0.247,148.101,0.467,-12.002,0.1700,0.126,Billie Eilish,4,0.0431,93,spotify:track:3QaPy1KgI7nu9FJEQUgn6h,3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,0.000271,0,6,261467,0.6120,music,"pop,global"
7,0.907,112.964,0.674,-4.086,0.2970,0.721,Sabrina Carpenter,4,0.0640,93,spotify:track:5G2f63n7IPVPPjfNIGih7Q,5G2f63n7IPVPPjfNIGih7Q,Taste,0.000000,1,3,157280,0.1010,music,"pop,global"
8,0.808,108.548,0.554,-4.169,0.1590,0.372,Gracie Abrams,4,0.0368,93,spotify:track:7ne4VBA60CxGM75vw0EYad,7ne4VBA60CxGM75vw0EYad,That’s So True,0.000000,1,1,166300,0.2140,music,"pop,global"
9,0.413,94.938,0.494,-10.432,0.1930,0.273,Gigi Perez,4,0.0254,93,spotify:track:2262bWmqomIaJXwCRHr13j,2262bWmqomIaJXwCRHr13j,Sailor Song,0.000067,1,11,211979,0.6820,music,"folk,indie"


In [74]:
df.to_csv(SPOTIFY_KAGGLE_REFINE_CSV, index=False, encoding="utf-8-sig")